## Configuration parameters
**ONLY CHANGE THE parameters here** and rerun the notebook

In [3]:
# I/O
pg_report_path: str = "../data/report.pg_matrix.tsv"
psm_report_path: str = "../data/report.tsv"
save_dir: str = "../output/"

## Setup

### Imports

In [ ]:
import anndata as ad
import mudata as md
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

In [ ]:
# load h5ad data
prec_adata = ad.read_h5ad("../data/albrecht.precursors.h5ad")
prot_adata = ad.read_h5ad("../data/albrecht.proteins.h5ad")

In [ ]:
# Build the precursor -> protein adjacency matrix (varp)
# Each precursor maps to one protein group via Protein.Group column
prot_index = pd.Index(prot_adata.var.index)
prec_protein_groups = prec_adata.var["Protein.Group"]

# Row indices: precursors, Col indices: proteins
row_idx = np.arange(len(prec_adata.var))
col_idx = prot_index.get_indexer(prec_protein_groups)

# Sanity check: all precursors should map to a known protein group
assert (col_idx >= 0).all(), "Some precursors don't map to any protein group!"

mapping_matrix = csr_matrix(
    (np.ones(len(row_idx), dtype=np.float32), (row_idx, col_idx)),
    shape=(len(prec_adata.var), len(prot_adata.var)),
)
print(f"Mapping matrix shape: {mapping_matrix.shape} (precursors x proteins)")
print(f"Non-zero entries: {mapping_matrix.nnz}")

In [15]:
msdata = md.MuData(
    # These are the raw data levels
    {
        "protein_level": prot_adata,
        #"peptide_level": ad.AnnData(...),
        "precursor_level": prec_adata,
    },
    # This stores the feature mapping as adjacency matrix of a DAG
    #varp = csr_matrix(...)
)

  self._update_attr("var", axis=0, join_common=join_common)

  self._update_attr("obs", axis=1, join_common=join_common)



In [14]:
?md.MuData

Init signature:
md.MuData(
    data: anndata._core.anndata.AnnData | collections.abc.Mapping[str, anndata._core.anndata.AnnData] | ForwardRef('MuData') | None = None,
    feature_types_names: dict | None = mappingproxy({'Gene Expression': 'rna', 'Peaks': 'atac', 'Antibody Capture': 'prot'}),
    as_view: bool = False,
    index: tuple[slice | numbers.Integral, slice | numbers.Integral] | slice | numbers.Integral | None = None,
    **kwargs,
)
Docstring:     
Multimodal data object

MuData represents modalities as collections of AnnData objects
as well as includes multimodal annotations
such as embeddings and neighbours graphs learned jointly
on multiple modalities and generalised sample
and feature metadata tables.

Parameters
----------
data
    AnnData object or dictionary with AnnData objects as values.
    If a dictionary is passed, the keys will be used as modality names.
    If None, creates an empty MuData object.
feature_types_names
    Dictionary to map feature types encoded i

In [9]:
msdata

MuData object with n_obs × n_vars = 1801 × 17250
  2 modalities
    protein_level:	1801 x 2161
    precursor_level:	1801 x 15089
      var:	'Run', 'Stripped.Sequence', 'Precursor.Charge', 'RT', 'RT.Start', 'RT.Stop', 'IM', 'Protein.Group', 'Protein.Ids', 'Genes', 'MS2.Scan', 'CScore', 'Q.Value', 'Global.Q.Value', 'Global.PG.Q.Value', 'Lib.Q.Value', 'Lib.PG.Q.Value'

In [16]:
prot_adata

AnnData object with n_obs × n_vars = 1801 × 2161

In [17]:
prec_adata

AnnData object with n_obs × n_vars = 1801 × 15089
    var: 'Run', 'Stripped.Sequence', 'Precursor.Charge', 'RT', 'RT.Start', 'RT.Stop', 'IM', 'Protein.Group', 'Protein.Ids', 'Genes', 'MS2.Scan', 'CScore', 'Q.Value', 'Global.Q.Value', 'Global.PG.Q.Value', 'Lib.Q.Value', 'Lib.PG.Q.Value'